In [13]:
import math
import numpy as np
import pandas as pd
import os
import csv
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader,random_split
from torchvision import transforms

In [14]:
def same_seed(seed):
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [15]:
device="cuda" if torch.cuda.is_available() else 'cpu'
config={
    'seed':5201314,
    'valid_ratio':0.2,
    'n_epochs':50,
    'batch_size':64,
    'learning_rate':1e-3,
    'early_stop':10,
    'save_path':'./digit_model/digit_cnn_model.ckpt'
}

In [16]:
def train_valid_split(raw_train_data,valid_ratio,seed):
    valid_data_size=int(len(raw_train_data)*valid_ratio)
    train_data_size=len(raw_train_data)-valid_data_size
    train_subset,valid_subset=random_split(raw_train_data,lengths=[train_data_size,valid_data_size],generator=torch.Generator().manual_seed(seed))
    train_data=raw_train_data[train_subset.indices]
    valid_data=raw_train_data[valid_subset.indices]
    return train_data,valid_data

In [17]:
def preprocess_data(train_data,valid_data,test_data):
    y_train=train_data[:,0]
    y_valid=valid_data[:,0]
    x_train=train_data[:,1:]
    x_valid=valid_data[:,1:]
    x_test=test_data
    x_train=x_train/255.0
    x_valid=x_valid/255.0
    x_test=x_test/255.0
    return x_train,x_valid,x_test,y_train,y_valid

In [18]:
class digit_dataset(Dataset):
    def __init__(self,features,targets=None,transform=None):
        if targets is None:
            self.targets=targets
        else:
            self.targets=torch.LongTensor(targets)
        self.features=torch.FloatTensor(features).reshape(-1,1,28,28)
        self.transforms=transform
    def __getitem__(self, index):
        if self.transforms is not None:
            return self.transforms(self.features[index]),self.targets[index]
        if self.targets is None:
            return self.features[index]
        else:
            return self.features[index],self.targets[index]

    def __len__(self):
        return len(self.features)

In [19]:
class res_block(nn.Module):
    def __init__(self, in_channels,out_channels,stride=1):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=stride,padding=1)
        self.bn1=nn.BatchNorm2d(out_channels)
        self.relu=nn.ReLU()
        self.conv2=nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1)
        self.bn2=nn.BatchNorm2d(out_channels)
        if in_channels!=out_channels or stride!=1:
            self.shortcup=nn.Sequential(
                nn.Conv2d(in_channels,out_channels,kernel_size=1,stride=stride),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcup=nn.Sequential()

    def forward(self,x):
        out=self.conv1(x)
        out=self.bn1(out)
        out=self.relu(out)
        out=self.conv2(out)
        out=self.bn2(out)
        out=out+self.shortcup(x)
        out=self.relu(out)
        return out

In [20]:
class digit_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre_layer=nn.Sequential(
            nn.Conv2d(1,16,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.layer_1=res_block(in_channels=16,out_channels=16)
        self.pool_1=nn.MaxPool2d(kernel_size=2,stride=2)
        self.layer_2=res_block(in_channels=16,out_channels=32)
        self.fc_layer=nn.Sequential(
            nn.Linear(32*14*14,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128,10)
        )

    def forward(self,x):
        x=self.pre_layer(x)
        x=self.layer_1(x)
        x=self.pool_1(x)
        x=self.layer_2(x)
        x=x.flatten(1)
        x=self.fc_layer(x)
        return x

In [21]:
def digit_trainer(train_loader,valid_loader,config,model,device):
    criterion=nn.CrossEntropyLoss()
    optimizer=torch.optim.AdamW(model.parameters(),lr=config['learning_rate'],weight_decay=1e-4)
    if not os.path.isdir('./digit_model'):
        os.mkdir('./digit_model')
    n_epochs=config['n_epochs']
    best_loss=math.inf
    step=0
    early_stop_count=0

    for epoch in range(n_epochs):
        model.train()
        loss_record=[]
        train_pbar=tqdm(train_loader,position=0,leave=False)
        for x,y in train_pbar:
            x,y=x.to(device),y.to(device)
            optimizer.zero_grad()
            pred=model(x)
            loss=criterion(pred,y)
            loss.backward()
            optimizer.step()
            step+=1
            loss_record.append(loss.detach().item())
            train_pbar.set_description(f'Epoch {epoch+1}/{n_epochs}')
        mean_train_loss=sum(loss_record)/len(loss_record)

        model.eval()
        loss_record=[]
        acc_record=[]
        for x,y in valid_loader:
            x,y=x.to(device),y.to(device)
            with torch.no_grad():
                pred=model(x)
                loss=criterion(pred,y)
                pred_class=torch.argmax(pred,dim=1)
                acc=(pred_class==y).float().mean()
                loss_record.append(loss.item())
                acc_record.append(acc.item())
        mean_valid_loss=sum(loss_record)/len(loss_record)
        mean_valid_acc=sum(acc_record)/len(acc_record)
        print(f'Epoch: {epoch+1}/{n_epochs}, train loss: {mean_train_loss:.4f}, valid loss: {mean_valid_loss:.4f}, valid acc: {mean_valid_acc:.4f}')
        if mean_valid_loss<best_loss:
            best_loss=mean_valid_loss
            torch.save(model.state_dict(),config['save_path'])
            print(f'saving model with {best_loss:.3f}')
            early_stop_count=0
        else:
            early_stop_count+=1

        if early_stop_count>=config['early_stop']:
            print('\nmodel is not improving, so we halt the training session.')
            return


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


main

In [ ]:
same_seed(config['seed'])
train_data=pd.read_csv('/content/drive/MyDrive/digit_train.csv').values
test_data=pd.read_csv('/content/drive/MyDrive/digit_test.csv').values
train_data,valid_data=train_valid_split(train_data,config['valid_ratio'],config['seed'])
x_train,x_valid,x_test,y_train,y_valid=preprocess_data(train_data,valid_data,test_data)
train_transform=transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0,scale=(0.9,1.1))
    ])
train_dataset=digit_dataset(x_train,y_train,transform=train_transform)
valid_dataset=digit_dataset(x_valid,y_valid)
test_dataset=digit_dataset(x_test)
train_loader=DataLoader(train_dataset,batch_size=config['batch_size'],shuffle=True,pin_memory=True)
valid_loader=DataLoader(valid_dataset,batch_size=config['batch_size'],shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=config['batch_size'],shuffle=False,pin_memory=True)
model=digit_model().to(device)
digit_trainer(train_loader,valid_loader,config,model,device)

Epoch: 1/50, train loss: 0.2018, valid loss: 0.0753, valid acc: 0.9760
saving model with 0.075


Epoch 2/50:  74%|███████▍  | 388/525 [00:29<00:09, 14.81it/s]

In [ ]:
def predict(test_loader,model,device):
    model.eval()
    preds=[]
    for x in tqdm(test_loader):
        x=x.to(device)
        with torch.no_grad():
            pred=model(x)
            pred_class=torch.argmax(pred,dim=1)
            preds.append(pred_class.detach().cpu())
    preds=torch.cat(preds,dim=0).numpy()
    return preds

def save_pred(preds,file):
    with open(file,'w') as fp:
        writer=csv.writer(fp)
        writer.writerow(['ImageId','Label'])
        for i,p in enumerate(preds,start=1):
            writer.writerow([i,p])

model=digit_model().to(device)
model.load_state_dict(torch.load(config['save_path']))
preds=predict(test_loader,model,device)
save_pred(preds,'digit_submission.csv')